In [1]:
import copy
import json
from pathlib import Path

import pandas as pd
import torch
import plotly.express as px

from pivotal_tokens.hf.loading import load_model, load_tokenizer
from pivotal_tokens.constants import get_data_dir, get_artifacts_dir
from pivotal_tokens.extractor import THINKING_START_TOKEN, THINKING_END_TOKEN, TRUNCATED_TAG


# EXP41_DIR = get_data_dir() / "experiments" / "exp4.1-likelihood-tokens" / "exp4.1.1-qwen3-1.7b-likelihood-tokens"
# EXP41_DIR = get_data_dir() / "experiments" / "exp4.1-likelihood-tokens" / "exp4.1.2-qwen3-1.7b-likelihood-tokens-trunc"
EXP41_DIR = get_data_dir() / "experiments" / "exp4.1-likelihood-tokens" / "exp4.1.3-qwen3-1.7b-likelihood-tokens-trunc-v2"
EXP41_RESULTS_FILE = EXP41_DIR / "data" / "loglikelihood_pivotal_tokens.csv"

EXP22_DIR = get_data_dir() / "experiments" / "exp2.2-alt-tokens-on-pivotal-positions" / "exp2.2.1-qwen3-1.7b-alt-tokens-on-pivotal-positions"
EXP22_RESULTS_FILE = EXP22_DIR / "data" / "alternative_tokens.csv"

TRACES_FILE = get_artifacts_dir() / 'exp1.1.1-qwen3-1.7b-traces.csv'

# PREPROCESSED_FILE = Path("data/preprocessed.csv")
# PREPROCESSED_FILE = Path("data/preprocessed-trunc.csv")
PREPROCESSED_FILE = Path("data/preprocessed-trunc-v2.csv")


QWEN3_1_7B_MODEL_ID = 'Qwen/Qwen3-1.7B'
DEVICE = "cuda"


torch.set_grad_enabled(False)

In [2]:
# tokenizer = load_tokenizer(QWEN3_1_7B_MODEL_ID)

# prob_init_df = pd.read_csv(EXP22_RESULTS_FILE)
# prob_init_df = prob_init_df[["sample_id", "prob_init"]].rename(columns={"sample_id": "id"}).copy()
# traces_df = pd.read_csv(TRACES_FILE)
# traces_df = pd.merge(traces_df, prob_init_df, on="id", how="inner")


# nnll_df = pd.read_csv(EXP41_RESULTS_FILE)
# nnll_df['token_ids'] = nnll_df['token_ids'].apply(json.loads)

# df = pd.merge(traces_df, nnll_df, left_on='id', right_on='sample_id', how='inner')

In [3]:
# def get_thinking_trace_from_completion(completion: str) -> str:
#     prefix_split_str = '<|im_start|>assistant\n'
#     completion_with_suffix = completion.split(prefix_split_str)[-1].strip()

#     suffix_split_str = '<|im_end|>'
#     completion = completion_with_suffix.split(suffix_split_str)[0].strip()
#     return completion


# def get_token_length(text: str, tokenizer: PreTrainedTokenizer) -> int:
#     token_ids = tokenizer.encode(text, add_special_tokens=False)
#     return len(token_ids)


# df["partial_trace"] = df["pivotal_context"].apply(get_thinking_trace_from_completion)
# df["partial_trace_token_length"] = df["partial_trace"].apply(
#     lambda x: get_token_length(x, tokenizer)
# )
# df["trace_token_length"] = df["trace"].apply(lambda x: get_token_length(x, tokenizer))
# df["pivotal_token_relative_position"] = (
#     (df["partial_trace_token_length"] + 1) / df["trace_token_length"]
# )

# df = df.drop(columns=['pivotal_context'])
# df = df.drop(columns=['trace', 'metadata', 'partial_trace'])

# df.to_csv(PREPROCESSED_FILE, index=False)

In [4]:
def parse_token_ids(value: object) -> list[int]:
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return []
    return []


if PREPROCESSED_FILE.exists():
    prep_df = pd.read_csv(PREPROCESSED_FILE)
    prep_df = prep_df.drop_duplicates()
    prep_df["token_ids"] = prep_df["token_ids"].apply(parse_token_ids)
else:
    prep_df = df.copy()

In [5]:
model = load_model(model_id=QWEN3_1_7B_MODEL_ID, device=DEVICE, use_flash_attention=False)
tokenizer = load_tokenizer(model_id=QWEN3_1_7B_MODEL_ID)
model.eval()

print(f"Model loaded: {QWEN3_1_7B_MODEL_ID}")
print(f"Device: {next(model.parameters()).device}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen3-1.7B
Device: cuda:0


In [64]:
# Find samples with large nnll spike (high range between max and min nnll across token positions)
sample_stats = (
    prep_df.groupby("sample_id")["nnll"]
    .agg(["mean", "std", "max", "min", "count"])
    .reset_index()
)
sample_stats["range"] = sample_stats["max"] - sample_stats["min"]
sample_stats = sample_stats.sort_values("range", ascending=False)

print("Top samples by nnll range:")
print(sample_stats.head(10).to_string(index=False))

# Pick the top spike sample for debugging
# selected_sample_id = sample_stats.iloc[3]["sample_id"]
# selected_sample_id = "5a7ae2f2554299042af8f6aa"
# selected_sample_id = "5a7e801b5542997cc2c47557"
selected_sample_id = "5a738225554299623ed4abc3"
print(f"\nSelected sample_id: {selected_sample_id}")

sample_row = prep_df[prep_df["sample_id"] == selected_sample_id].iloc[0]
print(f"Query: {sample_row['query']}")
print(f"Ground truth: {sample_row['ground_truth']}")

Top samples by nnll range:
               sample_id     mean      std    max      min  count     range
5ae54b6355429908b63265cc 6.697679 2.821160 21.625 0.185547    262 21.439453
5a8834f05542994846c1ce26 6.209011 3.050499 23.000 1.585938    223 21.414062
5a82ffef55429954d2e2ebf2 4.817383 2.687102 21.000 0.458984    234 20.541016
5a89bcc85542992e4fca83a7 4.185200 1.968237 20.875 0.593750    620 20.281250
5adeda67554299728e26c7db 6.836004 1.821764 22.000 2.015625   1061 19.984375
5ade15ae5542990dbb2f7f4c 7.071763 3.065730 22.250 2.265625    140 19.984375
5adcd6705542992c1e3a2426 6.662535 2.255963 22.625 2.812500    266 19.812500
5a78d06355429974737f78b4 6.521777 3.012875 21.000 1.312500    207 19.687500
5a7ae2f2554299042af8f6aa 6.105933 2.668744 21.375 1.851562    286 19.523438
5ae7e1fc55429952e35ea9cc 6.571989 2.556390 21.375 2.281250    247 19.093750

Selected sample_id: 5a738225554299623ed4abc3
Query: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christma

In [65]:
SYSTEM_PROMPT = (
    'Answer the following question directly, strictly without chain-of-thought after "</think>". '
    'The reasoning trace may be truncated; if so, it will end with "<truncated>" placed right before '
    'the closing "</think>". In that case, use the partial reasoning to make your best guess at the answer. '
    'Keep the answer short (e.g., "yes" or "no" for binary questions, a person\'s full name if the question '
    'expects a person, a country name if it asks about a country, etc.). '
    'Output the answer strictly after the prefix "Answer: `<answer>`" with no extra text.'
)

traces_df = pd.read_csv(TRACES_FILE)
trace_row = traces_df[traces_df["id"] == selected_sample_id].iloc[0]
reasoning_trace = trace_row["trace"]

if not reasoning_trace.startswith(THINKING_START_TOKEN):
    reasoning_trace = f"{THINKING_START_TOKEN}\n{reasoning_trace}"
if reasoning_trace.endswith(THINKING_END_TOKEN):
    reasoning_trace = reasoning_trace[: -len(THINKING_END_TOKEN)].rstrip()

query = sample_row["query"]
expected_answer = sample_row["ground_truth"]

prompts = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": query}]
context = tokenizer.apply_chat_template(
    prompts, tokenize=False, add_generation_prompt=True, enable_thinking=True
)
answer_suffix = f"{TRUNCATED_TAG}\n{THINKING_END_TOKEN}\nAnswer: `{expected_answer}`"


print(f"Query: {query}")
print(f"Expected answer: {expected_answer}")
print(f"Reasoning trace tokens: {len(tokenizer.encode(reasoning_trace, add_special_tokens=False))}")
print(f"Answer suffix: {repr(answer_suffix)}")

Query: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
Expected answer: Bedknobs and Broomsticks
Reasoning trace tokens: 165
Answer suffix: '<truncated>\n</think>\nAnswer: `Bedknobs and Broomsticks`'


In [66]:
completion_tok = tokenizer(reasoning_trace, return_tensors="pt").to(model.device)
completion_len = completion_tok.input_ids.shape[1]

full_text = context + reasoning_trace
full_tok = tokenizer(full_text, return_tensors="pt", return_attention_mask=False).to(model.device)

# Indices start from the <thinking> token in order to get nnll of the "no thinking" scenario.
# Specifically, the case "... <thinking> </thinking> ..."
completion_start_idx = full_tok.input_ids.shape[1] - completion_len + 1
completion_end_idx = full_tok.input_ids.shape[1]
indices = list(range(completion_start_idx, completion_end_idx))

n_ground_truth_tokens = len(tokenizer.encode(expected_answer))

In [67]:
past_key_values = None
prev_t = 0
last_prefix_logits = None

In [68]:
tokens = []

for t in indices:
    new_tokens = full_tok.input_ids[:, prev_t:t]

    if past_key_values is None:
        out_prefix = model(input_ids=new_tokens,
                            use_cache=True)
    else:
        past_length = past_key_values[0][0].shape[2]
        seq_len = new_tokens.shape[1]
        total_len = past_length + seq_len

        position_ids = torch.arange(past_length, total_len, device=model.device).unsqueeze(0)
        attention_mask = torch.ones(1, total_len, device=model.device, dtype=torch.long)

        out_prefix = model(input_ids=new_tokens,
                            past_key_values=past_key_values,
                            position_ids=position_ids,
                            attention_mask=attention_mask,
                            use_cache=True)

    past_key_values = out_prefix.past_key_values
    prev_t = t

    prefix_ids = full_tok.input_ids[:, :t]
    prefix_str = tokenizer.decode(prefix_ids[0], skip_special_tokens=False)

    conditional_context = prefix_str + answer_suffix
    conditional_context_tok = tokenizer(conditional_context, return_tensors="pt").to(model.device)

    suffix_ids = conditional_context_tok.input_ids[:, t:]
    suffix_input = suffix_ids[:, :-1]

    out_suffix = model(input_ids=suffix_input, use_cache=False, past_key_values=copy.deepcopy(past_key_values))
    past_key_values = out_prefix.past_key_values

    prev_last_prefix_logits = last_prefix_logits
    last_prefix_logits = out_prefix.logits[:, -1, :]

    logits_for_loss = torch.cat([last_prefix_logits.unsqueeze(1),
                            out_suffix.logits],
                            dim=1)
    
    target = suffix_ids.reshape(-1).clone()
    target[:-(n_ground_truth_tokens + 1)] = -100
    target[-1] = -100

    nnll = torch.nn.functional.cross_entropy(
        input=logits_for_loss.reshape(-1, logits_for_loss.size(-1)),
        target=target,
        reduction="mean",
    ).item()

    token_id = prefix_ids[:, -1:]
    token = tokenizer.decode(token_id[0])
    
    # Get logprob of the `t` token by shifting logprobs by -1
    if prev_last_prefix_logits is None:
        logits = out_prefix.logits[:, -2, :]
    else:
        logits = prev_last_prefix_logits
    all_token_logprobs = torch.nn.functional.log_softmax(logits, dim=-1)
    token_logprob = all_token_logprobs.gather(-1, token_id).squeeze(-1).item()


    tokens.append({
        'idx': t,
        'token_id': token_id[0].item(),
        'token': token,
        'nnll': nnll,
        'norm_coeff': n_ground_truth_tokens,
        'logprob': token_logprob
    })

In [69]:
nnll_df = pd.DataFrame(tokens)
nnll_df[:2]

,idx,token_id,token,nnll,norm_coeff,logprob
0,156,151667,<think>,1.937454,7,-0.000142
1,157,198,\n,2.107133,7,0.000000


In [70]:
sample_row['query'], sample_row['ground_truth']

('Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?',
 'Bedknobs and Broomsticks')

In [71]:
fig = px.line(nnll_df, y='nnll', custom_data=['token', 'logprob'], )
fig.update_traces(hovertemplate=(
    "<b>NNLL:</b> %{y:.4f}<br>"
    "<b>Logprob:</b> %{customdata[1]}<br>"
    "<b>Span:</b> %{customdata[0]}<extra></extra>"
))

fig.update_xaxes(
    tickvals=list(range(0, len(nnll_df))),
    ticktext=nnll_df['token'].tolist()
)